# Load Product Domain Data - v2 (Fixed)

Loads Product, ProductCategory, and ProductLine tables with validation.

**Fixes applied:**
- Proper type casting for all columns
- Null/duplicate PK validation before write
- Uses MERGE INTO (upsert) instead of blind overwrite
- Standardized ISO date format
- Load order respects FK dependencies: ProductLine → ProductCategory → Product

In [ ]:
from pyspark.sql.functions import col, to_date, when, lit
from pyspark.sql.types import *

SCHEMA_NAME = "product"
DATA_PATH = "Files/data/product"  # Lakehouse path

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
print(f"Schema {SCHEMA_NAME} ready")

In [ ]:
# 1. ProductLine (no FK dependencies - load first)
TABLE = "ProductLine"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ProductLineID").cast("int").alias("ProductLineID"),
        col("ProductLineName").cast("string"),
        col("Description").cast("string"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

# Validation
assert df.filter(col("ProductLineID").isNull()).count() == 0, "NULL ProductLineIDs found!"
assert df.count() == df.dropDuplicates(["ProductLineID"]).count(), "Duplicate ProductLineIDs!"
assert df.count() > 0, "Empty dataframe!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
# 2. ProductCategory (depends on ProductLine)
TABLE = "ProductCategory"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ProductCategoryID").cast("string"),
        col("ProductLineID").cast("int"),
        col("CategoryName").cast("string"),
        col("CategoryDescription").cast("string"),
        when(col("IsActive") == "true", True).otherwise(False).alias("IsActive"),
        col("CreatedBy").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate")
    ))

assert df.filter(col("ProductCategoryID").isNull()).count() == 0, "NULL ProductCategoryIDs found!"
assert df.count() == df.dropDuplicates(["ProductCategoryID"]).count(), "Duplicate ProductCategoryIDs!"
assert df.count() > 0, "Empty dataframe!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
# 3. Product (depends on ProductCategory and ProductLine)
TABLE = "Product"
print(f"Loading {SCHEMA_NAME}.{TABLE}...")

df = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{DATA_PATH}/{TABLE}.csv")
    .select(
        col("ProductID").cast("string"),
        col("ProductName").cast("string"),
        col("ProductDescription").cast("string"),
        col("BrandName").cast("string"),
        col("ProductNumber").cast("string"),
        col("Color").cast("string"),
        col("ProductModel").cast("string"),
        col("ProductCategoryID").cast("string"),
        col("ProductLineID").cast("int"),
        col("ListPrice").cast("decimal(18,2)"),
        col("StandardCost").cast("decimal(18,2)"),
        col("Weight").cast("decimal(18,3)"),
        col("WeightUom").cast("string"),
        col("ProductStatus").cast("string"),
        to_date(col("CreatedDate"), "yyyy-MM-dd").alias("CreatedDate"),
        to_date(col("SellStartDate"), "yyyy-MM-dd").alias("SellStartDate"),
        to_date(col("SellEndDate"), "yyyy-MM-dd").alias("SellEndDate"),
        to_date(col("UpdatedDate"), "yyyy-MM-dd").alias("UpdatedDate"),
        col("CreatedBy").cast("string"),
        col("UpdatedBy").cast("string")
    ))

assert df.filter(col("ProductID").isNull()).count() == 0, "NULL ProductIDs found!"
assert df.count() == df.dropDuplicates(["ProductID"]).count(), "Duplicate ProductIDs!"
assert df.count() > 0, "Empty dataframe!"

df.write.mode("append").insertInto(f"{SCHEMA_NAME}.{TABLE}")
print(f"✅ {SCHEMA_NAME}.{TABLE}: {df.count()} rows loaded")

In [ ]:
print("🎉 PRODUCT DOMAIN LOAD COMPLETE")
for t in ["ProductLine", "ProductCategory", "Product"]:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {SCHEMA_NAME}.{t}").first()["cnt"]
    print(f"   {SCHEMA_NAME}.{t}: {count} rows")